# CekAjaYuk - Data Preparation
## Persiapan Dataset untuk Deteksi Iklan Lowongan Kerja Palsu

Notebook ini berisi proses persiapan data untuk training model ML/DL:
1. Import libraries
2. Load dan eksplorasi dataset
3. Preprocessing gambar
4. Feature extraction
5. Data augmentation
6. Split dataset

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

print(f"TensorFlow version: {tf.__version__}")
print(f"OpenCV version: {cv2.__version__}")

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

# Paths
DATASET_DIR = '../dataset'  # Real dataset directory
GENUINE_DIR = os.path.join(DATASET_DIR, 'genuine')
FAKE_DIR = os.path.join(DATASET_DIR, 'fake')
DATA_DIR = '../data'  # Processed data output
MODELS_DIR = '../models'

# Create output directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Check if real dataset exists
if not os.path.exists(GENUINE_DIR) or not os.path.exists(FAKE_DIR):
    print(f"❌ Real dataset not found at {DATASET_DIR}")
    print(f"Expected: {GENUINE_DIR} and {FAKE_DIR}")
else:
    print(f"✅ Real dataset found at {DATASET_DIR}")
    genuine_count = len([f for f in os.listdir(GENUINE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    fake_count = len([f for f in os.listdir(FAKE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"   Genuine images: {genuine_count}")
    print(f"   Fake images: {fake_count}")
    print(f"   Total images: {genuine_count + fake_count}")

In [ ]:
# Function to load and preprocess images
def load_and_preprocess_image(image_path, target_size=IMG_SIZE):
    """
    Load and preprocess image for training
    """
    try:
        # Load image
        image = cv2.imread(image_path)
        if image is None:
            return None
        
        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Resize image
        image = cv2.resize(image, target_size)
        
        # Normalize pixel values
        image = image.astype(np.float32) / 255.0
        
        return image
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

def extract_image_features(image):
    """
    Extract traditional computer vision features from image
    """
    features = []
    
    # Convert to grayscale for some features
    gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    
    # 1. Color features
    # Mean RGB values
    features.extend(np.mean(image, axis=(0, 1)))
    
    # Standard deviation of RGB values
    features.extend(np.std(image, axis=(0, 1)))
    
    # 2. Texture features using Local Binary Pattern
    from skimage.feature import local_binary_pattern
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    lbp_hist = lbp_hist.astype(float)
    lbp_hist /= (lbp_hist.sum() + 1e-7)  # Normalize
    features.extend(lbp_hist)
    
    # 3. Edge features
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / (edges.shape[0] * edges.shape[1])
    features.append(edge_density)
    
    # 4. Brightness and contrast
    brightness = np.mean(gray)
    contrast = np.std(gray)
    features.extend([brightness, contrast])
    
    return np.array(features)

print("Image processing functions defined!")

In [ ]:
# Function to create sample dataset (for demonstration)
def create_sample_dataset():
    """
    Create sample dataset structure for demonstration
    In real implementation, you would have actual images
    """
    print("Creating sample dataset structure...")
    
    # Create sample data info
    sample_data = {
        'genuine': [
            'genuine_job_1.jpg', 'genuine_job_2.jpg', 'genuine_job_3.jpg',
            'genuine_job_4.jpg', 'genuine_job_5.jpg', 'genuine_job_6.jpg',
            'genuine_job_7.jpg', 'genuine_job_8.jpg', 'genuine_job_9.jpg',
            'genuine_job_10.jpg'
        ],
        'fake': [
            'fake_job_1.jpg', 'fake_job_2.jpg', 'fake_job_3.jpg',
            'fake_job_4.jpg', 'fake_job_5.jpg', 'fake_job_6.jpg',
            'fake_job_7.jpg', 'fake_job_8.jpg', 'fake_job_9.jpg',
            'fake_job_10.jpg'
        ]
    }
    
    # Create sample CSV with metadata
    data_list = []
    for label, files in sample_data.items():
        for file in files:
            data_list.append({
                'filename': file,
                'label': label,
                'path': os.path.join(DATA_DIR, label, file)
            })
    
    df = pd.DataFrame(data_list)
    df.to_csv(os.path.join(DATA_DIR, 'dataset_info.csv'), index=False)
    
    print(f"Sample dataset info created with {len(df)} entries")
    return df

# Create sample dataset
dataset_df = create_sample_dataset()
print("\nDataset overview:")
print(dataset_df.head())
print(f"\nLabel distribution:")
print(dataset_df['label'].value_counts())

In [ ]:
# Data augmentation setup
def create_data_generators():
    """
    Create data generators for training and validation
    """
    # Training data augmentation
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest',
        validation_split=0.2
    )
    
    # Validation data (no augmentation)
    val_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.2
    )
    
    return train_datagen, val_datagen

train_datagen, val_datagen = create_data_generators()
print("Data generators created successfully!")

In [ ]:
# Feature extraction for traditional ML
def prepare_traditional_ml_features():
    """
    Prepare features for traditional ML algorithms (Random Forest)
    """
    print("Preparing features for traditional ML...")
    
    # For demonstration, create synthetic features
    # In real implementation, extract from actual images
    np.random.seed(42)
    
    # Generate synthetic features for genuine jobs
    genuine_features = []
    for i in range(100):  # 100 genuine samples
        # Simulate features that might indicate genuine job posts
        features = [
            np.random.normal(0.6, 0.1),  # Professional color scheme
            np.random.normal(0.7, 0.1),  # Text density
            np.random.normal(0.8, 0.1),  # Logo presence
            np.random.normal(0.5, 0.1),  # Contact info completeness
            np.random.normal(0.6, 0.1),  # Layout quality
            np.random.normal(0.4, 0.1),  # Urgency indicators (low for genuine)
            np.random.normal(0.7, 0.1),  # Spelling accuracy
            np.random.normal(0.6, 0.1),  # Image quality
        ]
        genuine_features.append(features)
    
    # Generate synthetic features for fake jobs
    fake_features = []
    for i in range(100):  # 100 fake samples
        # Simulate features that might indicate fake job posts
        features = [
            np.random.normal(0.3, 0.1),  # Poor color scheme
            np.random.normal(0.4, 0.1),  # Low text density
            np.random.normal(0.2, 0.1),  # No logo
            np.random.normal(0.2, 0.1),  # Incomplete contact info
            np.random.normal(0.3, 0.1),  # Poor layout
            np.random.normal(0.8, 0.1),  # High urgency indicators
            np.random.normal(0.4, 0.1),  # Poor spelling
            np.random.normal(0.3, 0.1),  # Low image quality
        ]
        fake_features.append(features)
    
    # Combine features and labels
    X = np.array(genuine_features + fake_features)
    y = np.array([1] * 100 + [0] * 100)  # 1 for genuine, 0 for fake
    
    # Feature names
    feature_names = [
        'color_scheme_quality',
        'text_density',
        'logo_presence',
        'contact_completeness',
        'layout_quality',
        'urgency_indicators',
        'spelling_accuracy',
        'image_quality'
    ]
    
    return X, y, feature_names

X, y, feature_names = prepare_traditional_ml_features()
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Feature names: {feature_names}")

In [ ]:
# Split data for training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled successfully!")

In [ ]:
# Save preprocessed data
import joblib

# Save the scaler
joblib.dump(scaler, os.path.join(MODELS_DIR, 'feature_scaler.pkl'))

# Save the processed data
np.save(os.path.join(DATA_DIR, 'X_train.npy'), X_train_scaled)
np.save(os.path.join(DATA_DIR, 'X_val.npy'), X_val_scaled)
np.save(os.path.join(DATA_DIR, 'X_test.npy'), X_test_scaled)
np.save(os.path.join(DATA_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(DATA_DIR, 'y_val.npy'), y_val)
np.save(os.path.join(DATA_DIR, 'y_test.npy'), y_test)

# Save feature names
with open(os.path.join(DATA_DIR, 'feature_names.txt'), 'w') as f:
    for name in feature_names:
        f.write(f"{name}\n")

print("Data preparation completed and saved!")
print(f"Files saved in: {DATA_DIR}")
print(f"Models will be saved in: {MODELS_DIR}")

In [ ]:
# Visualize data distribution
plt.figure(figsize=(15, 10))

# Plot feature distributions
for i, feature_name in enumerate(feature_names):
    plt.subplot(2, 4, i+1)
    
    # Separate genuine and fake samples
    genuine_values = X[y == 1, i]
    fake_values = X[y == 0, i]
    
    plt.hist(genuine_values, alpha=0.7, label='Genuine', bins=20, color='green')
    plt.hist(fake_values, alpha=0.7, label='Fake', bins=20, color='red')
    plt.title(feature_name.replace('_', ' ').title())
    plt.legend()
    plt.xlabel('Feature Value')
    plt.ylabel('Frequency')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'feature_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Feature distribution visualization saved!")